In [0]:
from pyspark.sql import functions as F

# Read the bronze table from the catalog (never from the raw file)
df_bronze = spark.table("workspace.bronze.transactions")

print("Bronze rows:", df_bronze.count())

In [0]:
df_clean = (
    df_bronze
    # Remove extra spaces and force lowercase: "FAILED" becomes "failed"
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    # Missing or empty category becomes "unknown"; otherwise keep the trimmed text
    .withColumn(
        "category",
        F.when(
            F.col("category").isNull() | (F.trim(F.col("category")) == ""),
            F.lit("unknown"),
        ).otherwise(F.trim(F.col("category"))),
    )
)

In [0]:
df_typed = (
    df_clean
    # Replace the decimal comma by a dot, then convert text to exact decimal
    .withColumn("amount", F.regexp_replace("amount", ",", ".").cast("decimal(12,2)"))
    # Convert text "2026-09-15" into a real date
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
)

In [0]:
# Check the new schema: amount must be decimal and transaction_date must be date
df_typed.printSchema()

# Look at the specific rows that had defects
display(
    df_typed
    .filter(F.col("transaction_id").isin("TX00011", "TX00031", "TX00041"))
    .select("transaction_id", "category", "status", "amount", "transaction_date")
)

# Status should now have only 3 values
display(df_typed.groupBy("status").count())

# No amount or date should have failed to convert
print("Null amounts:", df_typed.filter(F.col("amount").isNull()).count())
print("Null dates:", df_typed.filter(F.col("transaction_date").isNull()).count())

In [0]:
# Etapa 5.2: remover duplicatas e gravar a silver

In [0]:
from pyspark.sql import Window

# Group rows by transaction_id; inside each group, newest ingestion comes first
w = Window.partitionBy("transaction_id").orderBy(F.col("_ingested_at").desc())

df_silver = (
    df_typed                                  # was df_clean (text cleaned, but not typed)
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(df_silver.count())  # expected: 500
df_silver.printSchema()   # expected: amount decimal(12,2), transaction_date date

In [0]:
#Célula 6: gravar a tabela silver

#(
    #df_silver.write
    #.format("delta")           # same table format used in bronze
   # .mode("overwrite")         # study mode: rerun without duplicating data
    #.saveAsTable("workspace.silver.transactions")
#)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")   # allow column types to change
    .saveAsTable("workspace.silver.transactions")
)

In [0]:
#validar
silver = spark.table("workspace.silver.transactions")

print("rows:", silver.count())                              # expected: 500
print("distinct ids:", silver.select("transaction_id").distinct().count())  # expected: 500
silver.printSchema()
